In [1]:
from MY_CONV_DRIVER_NO_QUANT import NPUDriver # , LayerConfig
import numpy as np


# NPUDriver instance
driver = NPUDriver("no_quant_1.bit")

In [2]:
input_image = np.load('npy_files/input.npy')
weights = {
    'conv1': np.load('npy_files/layer1_0_weight.npy'),
    'conv2': np.load('npy_files/layer2_0_weight.npy'),
    'conv3': np.load('npy_files/layer3_0_weight.npy'),
    'conv4': np.load('npy_files/layer4_0_weight.npy'),
    'fc': np.load('npy_files/fc1_weight.npy')
}

In [3]:
def quantize_8bit(x):
    # PyTorch tensor를 numpy로 변환
    temp = x
    temp = np.floor(temp)
    # 9 LSB 제거 (오른쪽 시프트 9)
    output_shifted = np.right_shift(temp.astype(np.int32), 9)
    # 8-bit signed int 범위로 클리핑 (-128 ~ 127)
    quantized = np.clip(output_shifted, -128, 127).astype(np.int8)
    return quantized

In [4]:
def npu_golden_model_HW(input_image, weights):

    # Conv1 + Leaky ReLU
    x = driver.run_conv_2d(input_image, weights['conv1'], tile_h=26, tile_w=26, tile_oc=8)
    x = quantize_8bit(x)
    print("Conv1 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # Conv2 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv2'], tile_h=24, tile_w=24, tile_oc=16)
    x = quantize_8bit(x)
    print("Conv2 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # Conv3 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv3'], tile_h=22, tile_w=22, tile_oc=64)
    x = quantize_8bit(x)
    print("Conv3 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # Conv4 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv4'], tile_h=14, tile_w=14, tile_oc=16)
    x = quantize_8bit(x)
    print("Conv4 output shape:", x.shape)  # Debugging output
    x = driver.leaky_relu_loop(x)

    # MAX Pool
    x = driver.maxpool_loop(x, pool_size=2, stride=2)
    print("Maxpool output shape:", x.shape)  # Debugging output

    # FC
    output = driver.run_fc_2d(x, weights['fc'], tile_h=3, tile_w=3, tile_oc=1)
    output = quantize_8bit(output)

    return output.reshape(-1)

In [5]:
import time

start_time = time.time()
#############################################################
print("NPU가 이미지 0 ~ 4번 계산")
for i in range(5):
    output = npu_golden_model_HW(input_image[i], weights)
    print("출력 형태:", output.shape)       # (10,)
    print("출력 값:", output)
#############################################################
end_time = time.time()
runtime = end_time - start_time
print(f"Runtime: {runtime*1000:.3f}ms")

NPU가 이미지 0 ~ 4번 계산
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [ -53  -45   -8   53 -102  -51 -128  127   14   14]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [  39   28  127  -24  -35  -77   28 -128   11  -52]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [ -8  75   6 -40  25 -10 -24 -33  22 -31]
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [127 -94  16 -30 -73 -26  -2 -47 -12   5]
Conv1 output shape: (8, 26, 26)
Conv2 output 

In [6]:
answer = np.load('npy_files/output.npy')
print("정답 형태:", answer[0].shape)
for i in range(5):
    print("정답 레이블:", answer[i])

정답 형태: (10,)
정답 레이블: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]
정답 레이블: [  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]
정답 레이블: [ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]
정답 레이블: [127. -94.  16. -30. -72. -26.  -2. -47. -12.   5.]
정답 레이블: [-47. -50. -17. -69. 127. -13. -78. -12.  -2.  80.]


#### 아래부터는 Project1 CNN 아키텍쳐도 호환되는지 확인하는 코드

In [7]:
pj1_input_image = np.load('pj1_npy_files/input.npy')
pj1_weights = {
    'conv1': np.load('pj1_npy_files/layer1_0_weight.npy'),
    'conv2': np.load('pj1_npy_files/layer2_0_weight.npy'),
    'fc': np.load('pj1_npy_files/fc1_weight.npy')
}

In [8]:
def pj1_quantize_8bit(x):
    # PyTorch tensor를 numpy로 변환
    temp = x
#     temp = np.floor(temp)
    # 9 LSB 제거 (오른쪽 시프트 9)
    output_shifted = np.right_shift(temp.astype(np.int32), 10)
    # 8-bit signed int 범위로 클리핑 (-128 ~ 127)
    quantized = np.clip(output_shifted, -128, 127).astype(np.int8)
    return quantized

In [9]:
def pj1_npu_golden_model_HW(input_image, weights):

    # Conv1 + Leaky ReLU
    x = driver.run_conv_2d(input_image, weights['conv1'], tile_h=8, tile_w=8, tile_oc=8)
    x = pj1_quantize_8bit(x)
#     print("Conv1 output shape:", x.shape)  # Debugging output
    x = driver.relu_loop(x)

    # Conv2 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv2'], tile_h=8, tile_w=8, tile_oc=8)
    x = pj1_quantize_8bit(x)
#     print("Conv2 output shape:", x.shape)  # Debugging output
    x = driver.relu_loop(x)

    # MAX Pool
    x = driver.maxpool_loop(x, pool_size=2, stride=2)
#     print("Maxpool output shape:", x.shape)  # Debugging output

    # FC
    output = driver.run_fc_2d(x, weights['fc'], tile_h=3, tile_w=3, tile_oc=1)
    output = pj1_quantize_8bit(output)

    return output.reshape(-1)

In [10]:
import time

start_time = time.time()
#############################################################
print("NPU가 이미지 0 ~ 4번 계산")
for i in range(5):
    output = pj1_npu_golden_model_HW(pj1_input_image[i], pj1_weights)
    print("출력 형태:", output.shape)       # (10,)
    print("출력 값:", output)
#############################################################
end_time = time.time()
runtime = end_time - start_time
print(f"Runtime: {runtime*1000:.3f}ms")

NPU가 이미지 0 ~ 4번 계산
출력 형태: (10,)
출력 값: [ 0 -2  2  6 -5  9  1  4  3  3]
출력 형태: (10,)
출력 값: [11 -9  1  2 -2 -1  2  2 -1  5]
출력 형태: (10,)
출력 값: [-7 -4 -1  1  3 -4 -5  2  1 -3]
출력 형태: (10,)
출력 값: [-1  5  2  3  0  1  1  2  2  3]
출력 형태: (10,)
출력 값: [-2  1  0  0  3 -2 -2  4  3  7]
Runtime: 25833.756ms


In [11]:
answer = np.load('pj1_npy_files/output.npy')
print("정답 형태:", answer[0].shape)
for i in range(5):
    print("정답 레이블:", answer[i])

정답 형태: (10,)
정답 레이블: [ 0 -2  2  6 -5  9  1  4  3  3]
정답 레이블: [11 -9  1  2 -2 -1  2  2 -1  5]
정답 레이블: [-7 -4 -1  1  3 -4 -5  2  1 -3]
정답 레이블: [-1  5  2  3  0  1  1  2  2  3]
정답 레이블: [-2  1  0  0  3 -2 -2  4  3  7]


#### 여기부터는 Torch 스타일로 Class 구조를 설정

In [12]:
import numpy as np
from MY_CONV_DRIVER_NO_QUANT_v2 import Module, Conv2d, LeakyReLU, ReLU, MaxPool2d, Linear, NPUDriver

class AdvancedTargetNetwork(Module):
    def __init__(self, bitfile_path):
        super().__init__()
        self._driver = NPUDriver(bitfile_path)
        
        # Convolutional Layer 1
        self.conv1 = Conv2d(1, 8)  # kernel_size=3, padding=0 고정
        self.leaky_relu1 = LeakyReLU(alpha=0.25)
        
        # Convolutional Layer 2
        self.conv2 = Conv2d(8, 16)  # kernel_size=3, padding=0 고정
        self.leaky_relu2 = LeakyReLU(alpha=0.25)

        # Convolutional Layer 3
        self.conv3 = Conv2d(16, 64)  # kernel_size=3, padding=0 고정
        self.leaky_relu3 = LeakyReLU(alpha=0.25)

        # Convolutional Layer 4
        self.conv4 = Conv2d(64, 128)  # kernel_size=3, padding=0 고정
        self.leaky_relu4 = LeakyReLU(alpha=0.25)

        # Max Pooling Layer
        self.pool = MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layer
        self.fc = Linear(128 * 10 * 10, 10)
        
        # 드라이버를 모든 자식 계층에 등록
        self.register_driver(self._driver)


    def quantize_8bit(self, x):
        temp = np.floor(x)
        output_shifted = np.right_shift(temp.astype(np.int32), 9)
        quantized = np.clip(output_shifted, -128, 127).astype(np.int8)
        return quantized

    def forward(self, x):
        """입력 x: (channels, height, width)"""
        outputs = []
        

        single_input = x  # (channels, height, width)

        # Fmap 1 (28x28 -> 26x26)
        x_out = self.conv1.forward(single_input)
        x_out = self.quantize_8bit(x_out)
        x_out = self.leaky_relu1.forward(x_out)

        # Fmap 2 (26x26 -> 24x24)
        x_out = self.conv2.forward(x_out)
        x_out = self.quantize_8bit(x_out)
        x_out = self.leaky_relu2.forward(x_out)

        # Fmap 3 (24x24 -> 22x22)
        x_out = self.conv3.forward(x_out)
        x_out = self.quantize_8bit(x_out)
        x_out = self.leaky_relu3.forward(x_out)

        # Fmap 4 (22x22 -> 20x20)
        x_out = self.conv4.forward(x_out)
        x_out = self.quantize_8bit(x_out)
        x_out = self.leaky_relu4.forward(x_out)

        # After MaxPool (20x20 -> 10x10)
        x_out = self.pool.forward(x_out)
        x_out = x_out.flatten()

        # Fully Connected
        x_out = self.fc.forward(x_out)
        x_out = self.quantize_8bit(x_out)
        
        return x_out

    def load_weights(self, weight_files):
        """가중치 파일 로드"""
        self.conv1.set_weights(np.load(weight_files['conv1']).astype(np.float32))
        self.conv2.set_weights(np.load(weight_files['conv2']).astype(np.float32))
        self.conv3.set_weights(np.load(weight_files['conv3']).astype(np.float32))
        self.conv4.set_weights(np.load(weight_files['conv4']).astype(np.float32))
        self.fc.set_weights(np.load(weight_files['fc']).astype(np.float32))



In [13]:
# 모델 초기화
model = AdvancedTargetNetwork("no_quant_1.bit")

# 가중치 로드
weight_files = {
    'conv1': 'npy_files/layer1_0_weight.npy',
    'conv2': 'npy_files/layer2_0_weight.npy',
    'conv3': 'npy_files/layer3_0_weight.npy',
    'conv4': 'npy_files/layer4_0_weight.npy',
    'fc': 'npy_files/fc1_weight.npy'
}
model.load_weights(weight_files)

# 입력 데이터 및 레이블 로드
input_data = np.load('npy_files/input.npy')  # Shape: (10000, 1, 28, 28)

output = model.forward(input_data[0])
print(output)

[ -53  -45   -8   53 -102  -51 -128  127   14   14]


#### 조금 더 Torch 스타일에 가깝게

In [14]:
import numpy as np
import npu_driver.nn as nn

class AdvancedTargetNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Convolutional Layer 1
        self.conv1 = nn.Conv2d(1, 8)  # kernel_size=3, padding=0 고정
        self.relu1 = nn.LeakyReLU(alpha=0.25)
        
        # Convolutional Layer 2
        self.conv2 = nn.Conv2d(8, 16)  # kernel_size=3, padding=0 고정
        self.relu2 = nn.LeakyReLU(alpha=0.25)

        # Convolutional Layer 3
        self.conv3 = nn.Conv2d(16, 64)  # kernel_size=3, padding=0 고정
        self.relu3 = nn.LeakyReLU(alpha=0.25)

        # Convolutional Layer 4
        self.conv4 = nn.Conv2d(64, 128)  # kernel_size=3, padding=0 고정
        self.relu4 = nn.LeakyReLU(alpha=0.25)

        # Max Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layer
        self.fc = nn.Linear(128 * 10 * 10, 10)

    def quantize_8bit(self, x):
        temp = np.floor(x)
        output_shifted = np.right_shift(temp.astype(np.int32), 9)
        quantized = np.clip(output_shifted, -128, 127).astype(np.int8).astype(np.float32)
        return quantized

    def forward(self, x):
        """입력 x: (batch_size, channels, height, width) 또는 (channels, height, width)"""
        if len(x.shape) == 3:  # (channels, height, width)
            x = np.expand_dims(x, axis=0)  # (1, channels, height, width)
        
        batch_size = x.shape[0]
        outputs = []
        
        for i in range(batch_size):
            single_input = x[i]  # (channels, height, width)
            
            # Fmap 1 (28x28 -> 26x26)
            x_out = self.conv1.forward(single_input)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu1.forward(x_out)
            
            # Fmap 2 (26x26 -> 24x24)
            x_out = self.conv2.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu2.forward(x_out)
            
            # Fmap 3 (24x24 -> 22x22)
            x_out = self.conv3.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu3.forward(x_out)
            
            # Fmap 4 (22x22 -> 20x20)
            x_out = self.conv4.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu4.forward(x_out)
            
            # After MaxPool (20x20 -> 10x10)
            x_out = self.pool.forward(x_out)
            x_out = x_out.flatten()
            
            # Fully Connected
            x_out = self.fc.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            
            outputs.append(x_out)
        
        return np.array(outputs)

    def load_weights(self, weight_files):
        """가중치 파일 로드"""
        self.conv1.set_weights(np.load(weight_files['conv1']).astype(np.float32))
        self.conv2.set_weights(np.load(weight_files['conv2']).astype(np.float32))
        self.conv3.set_weights(np.load(weight_files['conv3']).astype(np.float32))
        self.conv4.set_weights(np.load(weight_files['conv4']).astype(np.float32))
        self.fc.set_weights(np.load(weight_files['fc']).astype(np.float32))

# 사용 예시
def main():
    # 모델 초기화
    model = AdvancedTargetNetwork()
    
    # 가중치 로드
    weight_files = {
        'conv1': 'npy_files/layer1_0_weight.npy',
        'conv2': 'npy_files/layer2_0_weight.npy',
        'conv3': 'npy_files/layer3_0_weight.npy',
        'conv4': 'npy_files/layer4_0_weight.npy',
        'fc': 'npy_files/fc1_weight.npy'
    }
    model.load_weights(weight_files)
    
    # 입력 데이터 및 레이블 로드
    input_data = np.load('npy_files/input.npy')  # Shape: (10000, 1, 28, 28)
    labels = np.load('npy_files/label.npy')      # Shape: (10000,)

    # 추론 수행
    predictions = []
    for i in range(5):        # input_data.shape[0]
        input_batch = input_data[i]  # (1, 28, 28)
        output = model.forward(input_batch)  # (1, 10)
        print(output)
        predicted = np.argmax(output[0])
        predictions.append(predicted)
    
    # 정확도 계산
    correct = np.sum(np.array(predictions) == labels[0:5])
    total = len(labels[0:5])
    accuracy = (correct / total) * 100
    print(f"총 {total}개의 이미지에 대해 추론을 완료했습니다.")
    print(f"정답률: {accuracy:.2f}%")

In [15]:
main()

[[ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]]
[[  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]]
[[ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]]
[[127. -94.  16. -30. -73. -26.  -2. -47. -12.   5.]]
[[-47. -50. -17. -69. 127. -13. -78. -12.  -2.  80.]]
총 5개의 이미지에 대해 추론을 완료했습니다.
정답률: 100.00%
